# Лабораторная работа: Анализ и прогнозирование временного ряда

## Цель работы

Изучение основных методов анализа и прогнозирования временных рядов.

## Задание

В рамках данной лабораторной работы необходимо:

1.  Выбрать набор данных (датасет) для решения задачи прогнозирования временного ряда.
2.  Визуализировать временной ряд и его основные характеристики.
3.  Разделить временной ряд на обучающую и тестовую выборку.
4.  Произвести прогнозирование временного ряда с использованием следующих методов:
    * Один из авторегрессионных методов (ARMA, ARIMA, ...).
    * Метод символьной регрессии.
    * Два метода на выбор из семейства МГУА (один из линейных методов COMBI / MULTI + один из нелинейных методов MIA / RIA) с использованием *аналога* библиотеки `gmdh`.
5.  Визуализировать тестовую выборку и каждый из прогнозов.
6.  Оценить качество прогноза в каждом случае с помощью подходящей метрики.

## Установка необходимых библиотек

In [1]:
!pip install pandas matplotlib seaborn statsmodels pmdarima scikit-learn sympy deap gplearn

/Volumes/university/education/6 term/tmo/labs/lab+/venv/bin/pip: line 2: /Volumes/university/education/6 term/tmo/labs/lab+ 1/venv/bin/python3.11: No such file or directory
/Volumes/university/education/6 term/tmo/labs/lab+/venv/bin/pip: line 2: exec: /Volumes/university/education/6 term/tmo/labs/lab+ 1/venv/bin/python3.11: cannot execute: No such file or directory


## 1. Загрузка и первичный анализ данных

В качестве датасета выберем данные о ежедневных минимумах температуры в Мельбурне. Этот датасет хорошо подходит для демонстрации методов временных рядов.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка датасета
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-minimum-temperatures.csv"
df = pd.read_csv(url, header=0, index_col=0, parse_dates=True)
series = df.iloc[:, 0]  # берем первый столбец как Series


# Переименуем столбец для удобства
df.name = 'Temperature'

# Выведем первые несколько строк и информацию о данных
print("Первые 5 строк данных:")
print(df.head())

print("\nИнформация о данных:")
df.info()

print("\nОсновные статистики:")
print(df.describe())

# Визуализация временного ряда
plt.figure(figsize=(14, 7))
plt.plot(df)
plt.title('Ежедневные минимальные температуры в Мельбурне')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.grid(True)
plt.show()

# Визуализация скользящего среднего и стандартного отклонения (для оценки стационарности)
rolmean = df.rolling(window=30).mean()
rolstd = df.rolling(window=30).std()

plt.figure(figsize=(14, 7))
plt.plot(df, label='Исходный ряд')
plt.plot(rolmean, label='Скользящее среднее (30 дней)', color='red')
plt.plot(rolstd, label='Скользящее стандартное отклонение (30 дней)', color='green')
plt.title('Скользящее среднее и стандартное отклонение')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

# Декомпозиция временного ряда
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df, model='additive', period=365) # Годовая сезонность

plt.figure(figsize=(14, 10))
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

HTTPError: HTTP Error 404: Not Found

## 2. Разделение данных на обучающую и тестовую выборки

Для корректной оценки моделей, временной ряд разделяется на обучающую и тестовую выборки. Обычно, тестовая выборка берется из конца временного ряда.

In [ ]:
train_size = int(len(df) * 0.8)
train, test = df[0:train_size], df[train_size:len(df)]

print(f"Размер обучающей выборки: {len(train)}")
print(f"Размер тестовой выборки: {len(test)}")

plt.figure(figsize=(14, 7))
plt.plot(train.index, train, label='Обучающая выборка')
plt.plot(test.index, test, label='Тестовая выборка', color='orange')
plt.title('Разделение временного ряда на обучающую и тестовую выборки')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

## 3. Прогнозирование временного ряда

### 3.1. Авторегрессионные методы (ARIMA)

Используем `pmdarima`, которая автоматически подбирает оптимальные параметры `p, d, q` для модели ARIMA.

In [ ]:
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Построение модели ARIMA
model_arima = auto_arima(train, seasonal=True, m=7,  # m=7 для недельной сезонности
                         suppress_warnings=True, 
                         stepwise=True,
                         trace=True) # trace=True для вывода процесса подбора

print(model_arima.summary())

# Прогнозирование
forecast_arima = model_arima.predict(n_periods=len(test))
forecast_arima = pd.Series(forecast_arima, index=test.index)

# Оценка качества прогноза
rmse_arima = np.sqrt(mean_squared_error(test, forecast_arima))
mae_arima = mean_absolute_error(test, forecast_arima)

print(f"\nRMSE для ARIMA: {rmse_arima:.3f}")
print(f"MAE для ARIMA: {mae_arima:.3f}")

# Визуализация прогноза ARIMA
plt.figure(figsize=(14, 7))
plt.plot(train.index, train, label='Обучающая выборка')
plt.plot(test.index, test, label='Тестовая выборка', color='orange')
plt.plot(forecast_arima.index, forecast_arima, label='Прогноз ARIMA', color='green', linestyle='--')
plt.title('Прогноз временного ряда с использованием ARIMA')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

### 3.2. Метод символьной регрессии (Genetic Programming for Symbolic Regression)

Для символьной регрессии будем использовать библиотеку `gplearn`. Она позволяет находить математические выражения, которые наилучшим образом описывают данные. Для временных рядов обычно используются лаговые значения в качестве признаков.

In [ ]:
from gplearn.genetic import SymbolicRegressor
from sklearn.ensemble import RandomForestRegressor

# Создание лаговых признаков для символьной регрессии
def create_lagged_features(data, n_lags):
    df_lagged = pd.DataFrame(data)
    for i in range(1, n_lags + 1):
        df_lagged[f'lag_{i}'] = df_lagged['Temperature'].shift(i)
    df_lagged = df_lagged.dropna()
    return df_lagged

n_lags = 7 # Используем 7 предыдущих значений как признаки

df_lagged = create_lagged_features(df, n_lags)

X = df_lagged.drop('Temperature', axis=1)
y = df_lagged['Temperature']

# Разделение лаговых данных на обучающую и тестовую выборки
X_train, X_test = X[X.index <= train.index[-1]], X[X.index > train.index[-1]]
y_train, y_test = y[y.index <= train.index[-1]], y[y.index > train.index[-1]]

# Обучение модели символьной регрессии
est_gp = SymbolicRegressor(population_size=5000,
                           generations=20,
                           tournament_size=30,
                           stopping_criteria=0.01,
                           const_range=(-1.0, 1.0),
                           init_depth=(2, 6),
                           init_method='half and half',
                           function_set=('add', 'sub', 'mul', 'div', 'sqrt', 'log', 'abs', 'neg', 'inv'),
                           metric='mse',
                           parsimony_coefficient=0.01,
                           random_state=42,
                           verbose=1,
                           n_jobs=-1) # Используем все доступные ядра процессора

est_gp.fit(X_train, y_train)

print(f"\nНайденное уравнение: {est_gp._program}")

# Прогнозирование
forecast_sr = est_gp.predict(X_test)
forecast_sr = pd.Series(forecast_sr, index=y_test.index)

# Оценка качества прогноза
rmse_sr = np.sqrt(mean_squared_error(y_test, forecast_sr))
mae_sr = mean_absolute_error(y_test, forecast_sr)

print(f"\nRMSE для Символьной регрессии: {rmse_sr:.3f}")
print(f"MAE для Символьной регрессии: {mae_sr:.3f}")

# Визуализация прогноза Символьной регрессии
plt.figure(figsize=(14, 7))
plt.plot(train.index, train, label='Обучающая выборка')
plt.plot(y_test.index, y_test, label='Тестовая выборка', color='orange')
plt.plot(forecast_sr.index, forecast_sr, label='Прогноз Символьной регрессии', color='purple', linestyle='--')
plt.title('Прогноз временного ряда с использованием Символьной регрессии')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

### 3.3. Методы семейства МГУА (GMDH - Group Method of Data Handling)

Поскольку библиотека `gmdh` может быть несовместима с macOS или быть труднодоступной, мы используем ее аналог или схожие по принципу работы методы. В данном случае, можно рассмотреть подходы, основанные на Polynomial Regression или Multi-Layer Perceptrons, которые могут эмулировать некоторые аспекты GMDH, строя сложные нелинейные модели. В качестве замены `gmdh` будем использовать библиотеку `pygmdh` (если доступна и работает на macOS) или же более общие методы машинного обучения, которые могут быть настроены для выполнения схожих задач.

Если `pygmdh` не устанавливается/не работает, можно рассмотреть использование `sklearn.linear_model.HuberRegressor` или `sklearn.neural_network.MLPRegressor` в сочетании с созданием полиномиальных признаков, чтобы имитировать построение сложных нелинейных зависимостей, характерных для GMDH.

In [ ]:
# Попытка установки pygmdh (может потребовать дополнительных зависимостей)
!pip install pygmdh


#### 3.3.1. Аналог линейного метода COMBI / MULTI (Использование `LinearRegression` с полиномиальными признаками)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Создание лаговых признаков для линейной модели
X_train_lin, X_test_lin = X_train, X_test # Используем те же лаговые признаки, что и для Symbolic Regression

# Создание полиномиальных признаков
degree = 2 # Степень полинома
model_poly_lin = make_pipeline(PolynomialFeatures(degree), LinearRegression())

# Обучение модели
model_poly_lin.fit(X_train_lin, y_train)

# Прогнозирование
forecast_lin_gmdh_analog = model_poly_lin.predict(X_test_lin)
forecast_lin_gmdh_analog = pd.Series(forecast_lin_gmdh_analog, index=y_test.index)

# Оценка качества прогноза
rmse_lin_gmdh_analog = np.sqrt(mean_squared_error(y_test, forecast_lin_gmdh_analog))
mae_lin_gmdh_analog = mean_absolute_error(y_test, forecast_lin_gmdh_analog)

print(f"\nRMSE для Аналога COMBI/MULTI (Poly Reg): {rmse_lin_gmdh_analog:.3f}")
print(f"MAE для Аналога COMBI/MULTI (Poly Reg): {mae_lin_gmdh_analog:.3f}")

# Визуализация прогноза
plt.figure(figsize=(14, 7))
plt.plot(train.index, train, label='Обучающая выборка')
plt.plot(y_test.index, y_test, label='Тестовая выборка', color='orange')
plt.plot(forecast_lin_gmdh_analog.index, forecast_lin_gmdh_analog, label='Прогноз Аналог COMBI/MULTI', color='brown', linestyle='--')
plt.title('Прогноз временного ряда с использованием аналога COMBI/MULTI (Polynomial Regression)')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

#### 3.3.2. Аналог нелинейного метода MIA / RIA (Использование `MLPRegressor` - Многослойный перцептрон)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Масштабирование данных очень важно для MLP
pipeline_mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42, activation='relu', solver='adam'))
])

# Обучение модели
pipeline_mlp.fit(X_train_lin, y_train) # Используем те же лаговые признаки

# Прогнозирование
forecast_mlp_gmdh_analog = pipeline_mlp.predict(X_test_lin)
forecast_mlp_gmdh_analog = pd.Series(forecast_mlp_gmdh_analog, index=y_test.index)

# Оценка качества прогноза
rmse_mlp_gmdh_analog = np.sqrt(mean_squared_error(y_test, forecast_mlp_gmdh_analog))
mae_mlp_gmdh_analog = mean_absolute_error(y_test, forecast_mlp_gmdh_analog)

print(f"\nRMSE для Аналога MIA/RIA (MLPRegressor): {rmse_mlp_gmdh_analog:.3f}")
print(f"MAE для Аналога MIA/RIA (MLPRegressor): {mae_mlp_gmdh_analog:.3f}")

# Визуализация прогноза
plt.figure(figsize=(14, 7))
plt.plot(train.index, train, label='Обучающая выборка')
plt.plot(y_test.index, y_test, label='Тестовая выборка', color='orange')
plt.plot(forecast_mlp_gmdh_analog.index, forecast_mlp_gmdh_analog, label='Прогноз Аналог MIA/RIA (MLP)', color='red', linestyle='--')
plt.title('Прогноз временного ряда с использованием аналога MIA/RIA (MLPRegressor)')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

## 4. Сводная визуализация и сравнение метрик

Соберем все прогнозы на одном графике и сравним метрики для всех моделей.

In [ ]:
plt.figure(figsize=(16, 8))
plt.plot(train.index, train, label='Обучающая выборка', color='blue')
plt.plot(test.index, test, label='Тестовая выборка', color='orange', linewidth=2)
plt.plot(forecast_arima.index, forecast_arima, label='Прогноз ARIMA', color='green', linestyle=':', marker='o', markersize=3)
plt.plot(forecast_sr.index, forecast_sr, label='Прогноз Символьная регрессия', color='purple', linestyle:'-.', marker='x', markersize=3)
plt.plot(forecast_lin_gmdh_analog.index, forecast_lin_gmdh_analog, label='Прогноз Аналог COMBI/MULTI', color='brown', linestyle:'--', marker='s', markersize=3)
plt.plot(forecast_mlp_gmdh_analog.index, forecast_mlp_gmdh_analog, label='Прогноз Аналог MIA/RIA (MLP)', color='red', linestyle:'--', marker='^', markersize=3)

plt.title('Сравнение прогнозов различных моделей')
plt.xlabel('Дата')
plt.ylabel('Температура (°C)')
plt.legend()
plt.grid(True)
plt.show()

# Сводная таблица метрик
results = {
    'Модель': ['ARIMA', 'Символьная регрессия', 'Аналог COMBI/MULTI', 'Аналог MIA/RIA (MLP)'] moderation
}
results_df = pd.DataFrame(results)
print("\nСравнение метрик качества прогноза:")
print(results_df)